# v16.5 — aturan episode AF ≥ 30 detik: alarm palsu per jam vs episode AF yang terdeteksi

Rancangan dikunci sebelum aturan dihitung: [RANCANGAN.md](RANCANGAN.md). Logika di `episode.py`. Aturan dari definisi klinis
(episode AF ≥ 30 s), tidak disetel. Sistem: S1 = D-1 + 30 s (primer), S2 = (D-1 dan v3.1) + 30 s, R = v3.1 + 30 s.

In [1]:
import json, sys, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import episode as E
sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
t0 = time.time()
E.jalankan_edb()                       # dilewati bila cache EDB per rekaman lengkap
D = E.muat_semua()
R = {"data": {ds: {"window": int(len(d["y2"])), "window_AF": int(d["y2"].sum()), "rekaman": int(np.unique(d["rec"]).size),
                   "unit": int(np.unique(d["unit"]).size), "jam": float(len(d["y2"]) * d["langkah"] / 3600)} for ds, d in D.items()}}
R["data"]["EDB"]["dibuang_batas_AF"] = int(D["EDB"]["buang_batas"])
display(pd.DataFrame(R["data"]).T)

,window,window_AF,rekaman,unit,jam,dibuang_batas_AF
AFDB,165978.0,66978.0,23.0,23.0,230.525000,NaN
LTAFDB,1272328.0,706582.0,84.0,84.0,1767.122222,NaN
MIT-BIH,14152.0,1514.0,47.0,46.0,19.655556,NaN
CPSC 2021,321818.0,111464.0,1424.0,105.0,446.969444,NaN
NSRDB,294162.0,0.0,18.0,18.0,408.558333,NaN
EDB,126578.0,247.0,90.0,90.0,175.802778,2.0
Latihan fisik,10822.0,0.0,163.0,52.0,30.061111,NaN


## 1. Kendali mutu: tanpa aturan = angka episode versi sebelumnya

In [2]:
# tanpa aturan (durasi 0), palsu per jam dihitung dengan interval sebenarnya; v16.4 memakai metrik_episode v3.3 (window terpusat).
# Pemeriksaan: alarm per window tanpa aturan sama dengan v16.4 (NSRDB) dan v16.3 (LTAFDB)
r164 = json.loads((E.REPO / "cache/v16.4/ringkasan.json").read_text()) if (E.REPO / "cache/v16.4/ringkasan.json").exists() else None
a = float(D["NSRDB"]["d1"].mean())
print("NSRDB alarm D-1 per window:", round(a, 5), "| v16.4:", round(r164["nsrdb"]["alarm"]["D-1"]["alarm"], 5) if r164 else "(v16.4 belum selesai)")
if r164: assert abs(a - r164["nsrdb"]["alarm"]["D-1"]["alarm"]) < 1e-12
r163 = json.loads((E.REPO / "cache/v16.3/ringkasan.json").read_text())
print("kunci v16.3:", list(r163)[:6])

NSRDB alarm D-1 per window: 0.0564 | v16.4: 0.0564
kunci v16.3: ['hasil', 'jumlah', 'kriteria', 'lolos', 'vonis_replikasi', 'ambang']


## 2. Hasil per dataset, sistem, dan durasi

In [3]:
H = {ds: E.evaluasi(d) for ds, d in D.items()}
R["hasil"] = {ds: {s: {str(dur): {k: v for k, v in m.items() if k != "palsu_unit"} for dur, m in h.items()} for s, h in hs.items()}
              for ds, hs in H.items()}
rows = []
for ds, hs in H.items():
    for s, h in hs.items():
        for dur in (0, 30, 60, 120):
            m = h[dur]
            rows.append({"data": ds, "sistem": s, "durasi": dur, "palsu/jam": round(m["palsu_per_jam"], 3),
                         "Se ep ≥30 s": round(m["se_episode_30"], 3), "Se durasi": round(m["se_durasi"], 3), "+P ep": round(m["ppv_episode"], 3)})
tab = pd.DataFrame(rows)
display(tab[tab.durasi.isin([0, 30])].set_index(["data", "sistem", "durasi"]))

palsu/jam  Se ep ≥30 s  Se durasi  +P ep
data          sistem          durasi                                          
AFDB          D-1             0           1.839        0.995      0.968  0.876
                              30          0.117        0.927      0.908  0.985
              v3.1            0          11.437        0.999      0.973  0.567
                              30          1.592        0.947      0.911  0.854
              S2 (D-1 & v3.1) 0           1.295        0.993      0.944  0.941
                              30          0.099        0.886      0.838  0.991
LTAFDB        D-1             0           5.740        0.993      0.984  0.610
                              30          0.462        0.941      0.955  0.918
              v3.1            0          12.514        0.997      0.979  0.577
                              30          0.942        0.922      0.925  0.916
              S2 (D-1 & v3.1) 0           2.694        0.992      0.969  0.876
                              30          0.190        0.885      0.908  0.983
MIT-BIH       D-1             0           4.935        1.000      0.991  0.464
                              30          0.204        0.977      0.928  0.929
              v3.1            0          37.241        1.000      0.970  0.174
                              30          3.714        0.886      0.831  0.529
              S2 (D-1 & v3.1) 0           3.968        1.000      0.961  0.672
                              30          0.102        0.864      0.813  0.977
CPSC 2021     D-1             0           7.030        0.998      0.983  0.658
                              30          0.814        0.911      0.912  0.904
              v3.1            0          25.720        0.999      0.979  0.343
                              30          2.980        0.892      0.913  0.732
              S2 (D-1 & v3.1) 0           5.199        0.997      0.963  0.779
                              30          0.615        0.835      0.859  0.941
NSRDB         D-1             0          20.550          NaN        NaN  0.000
                              30          1.234          NaN        NaN  0.000
              v3.1            0          51.895          NaN        NaN  0.000
                              30          3.569          NaN        NaN  0.000
              S2 (D-1 & v3.1) 0          16.331          NaN        NaN  0.000
                              30          0.453          NaN        NaN  0.000
EDB           D-1             0           5.199        1.000      1.000  0.002
                              30          0.176        1.000      1.000  0.061
              v3.1            0          15.461        1.000      1.000  0.000
                              30          1.024        1.000      1.000  0.006
              S2 (D-1 & v3.1) 0           1.098        1.000      1.000  0.010
                              30          0.046        1.000      1.000  0.200
Latihan fisik D-1             0           5.622          NaN        NaN  0.000
                              30          0.133          NaN        NaN  0.000
              v3.1            0          40.018          NaN        NaN  0.000
                              30          8.815          NaN        NaN  0.000
              S2 (D-1 & v3.1) 0           4.258          NaN        NaN  0.000
                              30          0.033          NaN        NaN  0.000

## 3. Target produk (dikunci) dan hipotesis

In [4]:
AF_DS = ("AFDB", "LTAFDB", "MIT-BIH", "CPSC 2021")
NEG_DS = ("NSRDB", "EDB", "Latihan fisik")
target = {}
for s in H["NSRDB"]:
    p = {ds: H[ds][s][30]["palsu_per_jam"] for ds in NEG_DS}
    se = {ds: H[ds][s][30]["se_episode_30"] for ds in AF_DS}
    target[s] = dict(palsu=p, se=se, palsu_ok=all(v <= E.TARGET_PALSU for v in p.values()), se_ok=all(v >= E.TARGET_SE for v in se.values()))
    target[s]["memenuhi"] = target[s]["palsu_ok"] and target[s]["se_ok"]
R["target"] = target
display(pd.DataFrame({s: {**{f"palsu/jam {k}": round(v, 3) for k, v in t["palsu"].items()}, **{f"Se ep {k}": round(v, 3) for k, v in t["se"].items()},
                          "memenuhi target": t["memenuhi"]} for s, t in target.items()}))
h1 = E.boot_palsu(H["NSRDB"]["D-1"][30]["palsu_unit"], H["NSRDB"]["v3.1"][30]["palsu_unit"])
h1_edb = E.boot_palsu(H["EDB"]["D-1"][30]["palsu_unit"], H["EDB"]["v3.1"][30]["palsu_unit"])
h2 = H["LTAFDB"]["D-1"][0]["se_durasi"] - H["LTAFDB"]["D-1"][30]["se_durasi"]
R["H1"] = dict(nsrdb=h1, edb=h1_edb, lolos=bool(h1["p_kurang"] >= 0.95))
R["H2"] = dict(turun_se_durasi_ltafdb=float(h2), lolos=bool(h2 <= 0.05))
print("H1 (S1 vs R, palsu/jam NSRDB):", h1, "→", "lolos" if R["H1"]["lolos"] else "GAGAL")
print("   EDB:", h1_edb)
print("H2 (Se durasi LTAFDB D-1 tanpa aturan − S1):", round(h2, 4), "→", "lolos" if R["H2"]["lolos"] else "GAGAL")
ok = [s for s, t in target.items() if t["memenuhi"]]
pilih = min(ok, key=lambda s: target[s]["palsu"]["NSRDB"] + target[s]["palsu"]["EDB"]) if ok else None
R["terpilih"] = pilih
print("sistem yang memenuhi target:", ok, "→ terpilih:", pilih)

,D-1,v3.1,S2 (D-1 & v3.1)
palsu/jam NSRDB,1.234,3.569,0.453
palsu/jam EDB,0.176,1.024,0.046
palsu/jam Latihan fisik,0.133,8.815,0.033
Se ep AFDB,0.927,0.947,0.886
Se ep LTAFDB,0.941,0.922,0.885
Se ep MIT-BIH,0.977,0.886,0.864
Se ep CPSC 2021,0.911,0.892,0.835
memenuhi target,False,False,False


H1 (S1 vs R, palsu/jam NSRDB): {'delta': -2.335039875986701, 'ci': [-4.51529898972912, -0.9243698225422646], 'p_kurang': 1.0, 'ci_a': [0.9040538711057801, 1.564066991696002]} → lolos
   EDB: {'delta': -0.8475406468738645, 'ci': [-1.3047600900605802, -0.48290204357927746], 'p_kurang': 1.0, 'ci_a': [0.03945966373196556, 0.35425353646528807]}
H2 (Se durasi LTAFDB D-1 tanpa aturan − S1): 0.0286 → lolos
sistem yang memenuhi target: [] → terpilih: None


## 4. Kurva durasi (deskriptif, tidak dipakai memilih)

In [5]:
kurva = {s: {ds: {dur: (round(H[ds][s][dur]["palsu_per_jam"], 3), round(H[ds][s][dur]["se_episode_30"], 3)) for dur in E.DURASI}
              for ds in ("NSRDB", "EDB", "LTAFDB", "AFDB")} for s in H["NSRDB"]}
R["kurva"] = kurva
for s, d in kurva.items():
    print("==", s); display(pd.DataFrame(d).T)
pu = H["NSRDB"]["D-1"][30]["palsu_unit"]
R["nsrdb_per_subjek_S1"] = {k: v[0] / v[1] for k, v in pu.items()}
print("S1 NSRDB palsu/jam per subjek:", {k: round(v, 2) for k, v in R["nsrdb_per_subjek_S1"].items()})

== D-1


,0,30,60,120,360
NSRDB,"(20.55, nan)","(1.234, nan)","(0.039, nan)","(0.0, nan)","(0.0, nan)"
EDB,"(5.199, 1.0)","(0.176, 1.0)","(0.006, 1.0)","(0.0, 1.0)","(0.0, 1.0)"
LTAFDB,"(5.74, 0.993)","(0.462, 0.941)","(0.126, 0.592)","(0.038, 0.355)","(0.002, 0.161)"
AFDB,"(1.839, 0.995)","(0.117, 0.927)","(0.022, 0.692)","(0.004, 0.438)","(0.0, 0.19)"


== v3.1


,0,30,60,120,360
NSRDB,"(51.895, nan)","(3.569, nan)","(0.512, nan)","(0.11, nan)","(0.012, nan)"
EDB,"(15.461, 1.0)","(1.024, 1.0)","(0.279, 1.0)","(0.119, 1.0)","(0.034, 1.0)"
LTAFDB,"(12.514, 0.997)","(0.942, 0.922)","(0.17, 0.551)","(0.032, 0.323)","(0.006, 0.122)"
AFDB,"(11.437, 0.999)","(1.592, 0.947)","(0.396, 0.707)","(0.101, 0.466)","(0.003, 0.174)"


== S2 (D-1 & v3.1)


,0,30,60,120,360
NSRDB,"(16.331, nan)","(0.453, nan)","(0.005, nan)","(0.0, nan)","(0.0, nan)"
EDB,"(1.098, 1.0)","(0.046, 1.0)","(0.006, 1.0)","(0.0, 1.0)","(0.0, 1.0)"
LTAFDB,"(2.694, 0.992)","(0.19, 0.885)","(0.053, 0.52)","(0.011, 0.292)","(0.0, 0.109)"
AFDB,"(1.295, 0.993)","(0.099, 0.886)","(0.025, 0.618)","(0.004, 0.362)","(0.0, 0.119)"


S1 NSRDB palsu/jam per subjek: {'16265': 1.18, '16272': 0.69, '16273': 1.72, '16420': 1.19, '16483': 2.28, '16539': 0.33, '16773': 2.36, '16786': 0.29, '16795': 1.0, '17052': 2.45, '17453': 1.34, '18177': 0.96, '18184': 0.68, '19088': 1.4, '19090': 1.95, '19093': 1.46, '19140': 0.05, '19830': 0.9}


## 5. Kesimpulan

In [6]:
t = R["target"]
for s in t:
    print(f"{s}: palsu/jam NSRDB {t[s]['palsu']['NSRDB']:.2f}, EDB {t[s]['palsu']['EDB']:.2f}, latihan {t[s]['palsu']['Latihan fisik']:.2f}; "
          f"Se episode ≥30 s " + ", ".join(f"{k} {v:.2f}" for k, v in t[s]["se"].items()) + f" → {'MEMENUHI' if t[s]['memenuhi'] else 'tidak memenuhi'} target")
print(f"H1 {'lolos' if R['H1']['lolos'] else 'GAGAL'} (Δ palsu/jam NSRDB S1 − R = {R['H1']['nsrdb']['delta']:+.2f}); H2 {'lolos' if R['H2']['lolos'] else 'GAGAL'}.")
print("Sistem terpilih v16.5:", R["terpilih"] or "tidak ada yang memenuhi target")
R["lingkungan"] = Ev._lingkungan(); R["durasi_menit"] = (time.time() - t0) / 60
E.simpan_json(R, E.CACHE / "ringkasan.json"); print("tersimpan")

D-1: palsu/jam NSRDB 1.23, EDB 0.18, latihan 0.13; Se episode ≥30 s AFDB 0.93, LTAFDB 0.94, MIT-BIH 0.98, CPSC 2021 0.91 → tidak memenuhi target
v3.1: palsu/jam NSRDB 3.57, EDB 1.02, latihan 8.82; Se episode ≥30 s AFDB 0.95, LTAFDB 0.92, MIT-BIH 0.89, CPSC 2021 0.89 → tidak memenuhi target
S2 (D-1 & v3.1): palsu/jam NSRDB 0.45, EDB 0.05, latihan 0.03; Se episode ≥30 s AFDB 0.89, LTAFDB 0.89, MIT-BIH 0.86, CPSC 2021 0.83 → tidak memenuhi target
H1 lolos (Δ palsu/jam NSRDB S1 − R = -2.34); H2 lolos.
Sistem terpilih v16.5: tidak ada yang memenuhi target


tersimpan
